# 🚀 YOLOv8 口罩检测
> Face Mask Detection 数据集 | T4 GPU

**Shift+Enter 逐个运行，不要一次性全部运行**

## 1. 检查 GPU + 挂载 Google Drive

In [1]:
# 检查 GPU 是否可用（必须是 T4）
!nvidia-smi

Tue Jun  9 01:42:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 克隆仓库 + 安装依赖

In [3]:
# 克隆你的仓库 安装依赖
!git clone https://github.com/qianz7884-blip/yolov8-compare.git
%cd /content/yolov8-compare
!pip install -e .

!yolo checks

Cloning into 'yolov8-compare'...
remote: Enumerating objects: 1086, done.
remote: Counting objects: 100% (1086/1086), done.
remote: Compressing objects: 100% (854/854), done.
remote: Total 1086 (delta 220), reused 1078 (delta 216), pack-reused 0 (from 0)
Receiving objects: 100% (1086/1086), 2.50 MiB | 6.44 MiB/s, done.
Resolving deltas: 100% (220/220), done.
/content/yolov8-compare
Obtaining file:///content/yolov8-compare
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.61-0.editable-py3-none-any.whl size=23403 sha256=0ee56a5cdb155a0e5a82d289bef7351fd1d9e31454d72af63992d14a76711192
  Stored in directory: /tmp/pip-ephem-wheel-cache-tmwvbax6/wheels/f3/3e/14/0527f1e369997afc0a5ab932c8fda70d3a261af0a588523aae
Succes

In [9]:
# 安装依赖（requirements.txt 已修复）

!pip install -q kagglehub
print('✅ 依赖安装完成')

✅ 依赖安装完成


## 3. 下载数据集（Face Mask Detection）

In [4]:
import xml.etree.ElementTree as ET
import shutil, random
from pathlib import Path
import kagglehub

print('📥 下载 Face Mask Detection 数据集...')
src = Path(kagglehub.dataset_download('andrewmvd/face-mask-detection'))

# 创建 YOLO 目录结构（与 data/mask.yaml 的 path 对应）
dst = Path('./datasets/mask')
for split in ['train', 'val']:
    (dst / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dst / 'labels' / split).mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    'with_mask': 0,
    'without_mask': 1,
    'mask_weared_incorrect': 2
}

# 解析 XML 标注 → YOLO 格式
all_data = []
for xml_path in sorted((src / 'annotations').glob('*.xml')):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)

    objs = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_MAP:
            continue
        bbox = obj.find('bndbox')
        xc = ((float(bbox.find('xmin').text) + float(bbox.find('xmax').text)) / 2) / w
        yc = ((float(bbox.find('ymin').text) + float(bbox.find('ymax').text)) / 2) / h
        bw = (float(bbox.find('xmax').text) - float(bbox.find('xmin').text)) / w
        bh = (float(bbox.find('ymax').text) - float(bbox.find('ymin').text)) / h
        objs.append(f'{CLASS_MAP[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

    if objs:  # 只保留有标注的图片
        all_data.append({'filename': filename, 'objects': objs})

# 80/20 分割（固定种子，可复现）
random.seed(42)
random.shuffle(all_data)
n = int(len(all_data) * 0.8)

for split, data in [('train', all_data[:n]), ('val', all_data[n:])]:
    for item in data:
        stem = Path(item['filename']).stem
        img_src = src / 'images' / item['filename']
        if img_src.exists():
            shutil.copy2(img_src, dst / 'images' / split / item['filename'])
        lbl_path = dst / 'labels' / split / f'{stem}.txt'
        lbl_path.write_text('\n'.join(item['objects']) + ('\n' if item['objects'] else ''))
print(dst.resolve())

print(f'✅ 训练集 {n} 张,  验证集 {len(all_data) - n} 张')

📥 下载 Face Mask Detection 数据集...
Using Colab cache for faster access to the 'face-mask-detection' dataset.
/content/yolov8-compare/datasets/mask
✅ 训练集 682 张,  验证集 171 张


## 4. 训练（yolov8s）

In [16]:
%%writefile /content/yolov8-compare/datasets/mask/mask.yaml
path: /content/yolov8-compare/datasets/mask

train: images/train
val: images/val

names:
  0: with_mask
  1: without_mask
  2: mask_weared_incorrect

Overwriting /content/yolov8-compare/datasets/mask/mask.yaml


In [17]:

!cat -n /content/yolov8-compare/datasets/mask/mask.yaml

     1	path: /content/yolov8-compare/datasets/mask
     2	
     3	train: images/train
     4	val: images/val
     5	
     6	names:
     7	  0: with_mask
     8	  1: without_mask
     9	  2: mask_weared_incorrect


In [26]:
!pwd
import ultralytics
print(ultralytics.__file__)
import os
import requests
from PIL import Image

# 1. 强行创建缺失的 assets 文件夹
assets_dir = '/content/yolov8-compare/ultralytics/assets'
os.makedirs(assets_dir, exist_ok=True)

# 2. 尝试从网络下载官方的 bus.jpg
url = 'https://ultralytics.com/images/bus.jpg'
try:
    response = requests.get(url, timeout=5)
    if response.status_code == 200:
        with open(os.path.join(assets_dir, 'bus.jpg'), 'wb') as f:
            f.write(response.content)
        print("🎉 成功从官方渠道补全了缺失的 bus.jpg 文件！")
    else:
        raise Exception
except:
    # 3. 如果网络不好下载失败，直接本地生成一张灰色的假图片骗过系统
    img = Image.new('RGB', (640, 640), color='gray')
    img.save(os.path.join(assets_dir, 'bus.jpg'))
    print("🛠️ 网络下载失败，已现场为你生成了一张备用 bus.jpg 补丁！")

/content/yolov8-compare
/content/yolov8-compare/ultralytics/__init__.py
🎉 成功从官方渠道补全了缺失的 bus.jpg 文件！


In [ ]:
# with open('/content/yolo/train.py', 'r') as f:
#    text = f.read()

#text = text.replace(
#    "ckpt = torch.load(weights, map_location='cpu')",
#    "ckpt = torch.load(weights, map_location='cpu', weights_only=False)"
#)

#with open('/content/yolo/train.py', 'w') as f:
#    f.write(text)

#print("修改完成")
#!grep -n "torch.load(weights" /content/yolo/train.py

!yolo detect train \
model=yolov8s.pt \
data=/content/yolov8-compare/datasets/mask/mask.yaml \
epochs=50 \
imgsz=640 \
workers=2 \
project=/content/drive/MyDrive/yolov8 \
name=example
amp=False
# !python train.py \
#     --img 640 \
#     --batch 16 \
#     --epochs 50 \
#     --data data/mask.yaml \
#     --weights yolov8s.pt \
#     --cfg models/yolov8s.yaml \
#     --project /content/drive/MyDrive/yolov8 \
#     --name example \
#     --workers 2

New https://pypi.org/project/ultralytics/8.4.62 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolov8-compare/datasets/mask/mask.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi

## 5. 查看训练结果

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import os
# import matplotlib as mpl

# # ==========================================
# # 🌟 核心复刻代码：引入 Seaborn 样式 🌟
# # ==========================================
# try:
#     import seaborn as sns
#     sns.set_theme(style="darkgrid") # 完美的灰底白网格样式
# except ImportError:
#     print("正在安装 seaborn 库，请稍候...")
#     !pip install seaborn -q
#     import seaborn as sns
#     sns.set_theme(style="darkgrid")

# # 设置默认的中文字体和符号（防止乱码，如果不需要可以注释掉）
# # mpl.rcParams['font.sans-serif'] = ['SimHei']
# mpl.rcParams['axes.unicode_minus'] = False

# # ==========================================

# # 1. 设置你的真实数据路径（保持不变）
# data_dir = '/content/drive/MyDrive/yolov5-mask/exp-c2f'
# csv_path = os.path.join(data_dir, 'results.csv')

# # 2. 读取并清洗数据（保持不变）
# if not os.path.exists(csv_path):
#     print(f"❌ 错误：在 {data_dir} 中没有找到 results.csv，请确认路径是否正确。")
# else:
#     df = pd.read_csv(csv_path)
#     df.columns = df.columns.str.strip() # 去掉列名可能存在的空格

#     # 3. 定义官方标准图的 10 个指标（保持不变）
#     metrics = [
#         'train/box_loss', 'train/obj_loss', 'train/cls_loss', 'metrics/precision', 'metrics/recall',
#         'val/box_loss', 'val/obj_loss', 'val/cls_loss', 'metrics/mAP_0.5', 'metrics/mAP_0.5:0.95'
#     ]

#     # 兼容性检查：模糊匹配魔改版代码可能的列名改动（保持不变）
#     for m in metrics:
#         if m not in df.columns:
#             matched = [col for col in df.columns if m in col]
#             if matched:
#                 df.rename(columns={matched[0]: m}, inplace=True)

#     # 4. 开始绘制标准的 2行5列 网格图（应用 Seaborn 样式）
#     fig, axes = plt.subplots(2, 5, figsize=(18, 8), tight_layout=True)
#     axes = axes.ravel() # 展平网格矩阵方便循环绘制

#     for i, metric in enumerate(metrics):
#         if metric in df.columns:
#             # 完美的蓝色带点曲线 (results)
#             # 调整了 markersize 以在 100 轮下依然清晰可视
#             axes[i].plot(df[metric], marker='.', color='#1f77b4', label='results', markersize=3)

#             # 红色虚线平滑曲线 (smooth) - 保持原图的虚线风格
#             if len(df) > 5:
#                 smooth_val = df[metric].ewm(span=min(10, len(df))).mean()
#                 axes[i].plot(smooth_val, color='#d62728', linestyle=':', label='smooth', linewidth=1.5)

#             axes[i].set_title(metric, fontsize=10)

#             # 只在第二个子图（train/obj_loss）显示图例，保持原图风格
#             if i == 1:
#                 axes[i].legend(loc='upper right', frameon=True)
#         else:
#             axes[i].text(0.5, 0.5, 'No Data', ha='center', va='center')
#             axes[i].set_title(metric)

#     # 5. 保存并展示结果
#     # 重新命名为 results_replicated.png 以防覆盖之前的标准图
#     save_path = os.path.join(data_dir, 'results.png')
#     plt.savefig(save_path, dpi=300)
#     plt.show()
#     print(f"🎉 成功！你想要完美的“灰底白网格”复刻图已生成，并保存在：\n👉 {save_path}")




# # from IPython.display import Image, display

# # exp = '/content/drive/MyDrive/yolov5-mask/exp-c2f'


# # print('📈 训练曲线')
# # display(Image(filename=f'{exp}/results.png'))

# # print('📊 混淆矩阵')
# # display(Image(filename=f'{exp}/confusion_matrix.png'))

# # print('🔍 验证样张')
# # display(Image(filename=f'{exp}/val_batch0_pred.jpg'))

## 6. 推理检测（上传图片）

In [ ]:
# from google.colab import files
# import glob

# uploaded = files.upload()
# for fname in uploaded.keys():
#     !python detect.py \
#         --weights /content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt \
#         --source {fname} \
#         --conf 0.25
#     result = glob.glob(f'/content/yolo/runs/detect/*/{fname}')
#     if result:
#         display(Image(filename=result[0]))

## 7. 下载模型

In [ ]:
# from google.colab import files
# files.download('/content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt')